# 12 · M1.7 — M1.5 with a smarter drift: the recent trend, kept only when it is significant

## Executive summary (read this first)

**M1.7 = M1.5 with one change: the drift.** Everything else (transform per asset type, Student-t shocks with ν fitted, EWMA sd
halflife 63, correlation from 260 steps) is M1.5.

| asset | M1.5 drift per step | **M1.7 drift per step** |
|---|---|---|
| rates, FX, equity factors, unemployment | 0 | m × max(0, 1 − 4 ÷ t²), m = mean of the **last 126 steps** |
| CPI, payrolls | mean of the last 260 steps | mean of the **last 12 steps** |

t = m ÷ (sd ÷ √126) is the t-statistic of the 6-month trend. This is **James–Stein shrinkage**, the standard way to use a noisy
mean: a trend that could be noise (|t| ≤ 2) counts for nothing; a strong one is kept (|t| = 3 keeps 56 %, |t| = 4 keeps 75 %).

**Why.** M1.5's "no drift" won on average, but it lost to M0 wherever a trend was steady: the crawling CNY peg of 2005–08
(1.98 × M0), the hiking cycles of 2004–06 and 2016–18 (1.29, 1.43). On CPI, the 260-month average (about 2.3 % a year) is
21 years old; the last 12 months say what inflation is now.

**Results** (§3): M1.7 **0.9312** × M0 (M1.5 0.9444); F1 0.883, F2 0.966, F3 0.912, F4 0.951. F2 gains most (0.990 → 0.966).
The constant 4 and the 126-step window were chosen on this backtest among a few: in scratch tests, constants 2, 4, 9 and windows
63, 126, 252 gave 0.931–0.942. So the number is somewhat optimistic.

§2 works the arithmetic through on four dates. M1.7 keeps a drift on 5–8 % of the backtest dates of rates, FX and equity
factors (§2's last table): mostly it still says "no trend". No realized outcome is read.

## 0 — Setup (as notebook 07)

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model: notebook 09's M1.5, with the recent trend as drift

`m17_draws(U, origin)` is M1.7; `drift17=False` is M1.5. Checks: (1) `drift17=False` gives notebook 09's draws exactly;
(2) it reproduces notebook 09's saved scores.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

def m13_draws(U, origin, steps_window=WINDOW, drift_rule=True, student_t=True, ewma_halflife=None):
    """ewma_halflife=None: the plain sd of the window (M1.3); a number: the EWMA sd with that halflife (M1.5)."""
    by_asset, x, log_mode, last = fit(U, origin, steps_window)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    if ewma_halflife is not None:                       # M1.5: EWMA sd per asset; correlation from the window
        sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
        sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])
        sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    def centre(a, h):
        anchor = 0.0 if log_mode[a] else last[a]
        use_drift = U["rules"][a]["drift"] if drift_rule else True
        return anchor + (s[h] * mu[ai[a]] if use_drift else 0.0)
    mean = np.array([centre(a, h) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((m0.N_DRAWS, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"]) if student_t else None
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(m0.N_DRAWS, 1))), 1.0)
    sm = mean + dev
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

# Every asset's whole history as steps (its own transform, M0's gap rule), computed once: the recent trend (M1.7) looks
# back 126 steps, the variance ratio (M1.8) over the whole history.
FULL = {}
for _u, _U in UNITS_DATA.items():
    for _a in _U["assets"]:
        _rows = _U["rows"][_a]
        _lm = _U["rules"][_a]["transform"] == "log_diff" and min(v for _, v in _rows) > 0
        _st, _ = m0._steps([(d, np.log(v)) for d, v in _rows] if _lm else _rows, "level" if _lm else _U["target_type"])
        _ds = sorted(_st)
        FULL[(_u, _a)] = (np.array([str(d) for d in _ds]), np.array([_st[d] for d in _ds]))

def hist_steps(U, a, origin):
    """All of one asset's steps up to the origin, oldest first."""
    d, v = FULL[(U["unit"], a)]
    return v[: np.searchsorted(d, origin, side="right")]

def recent_trend(U, a, origin, mu_window):
    """M1.7's drift per step.
    CPI, NFP (drift by type): the mean of the last 12 monthly steps -- this year's trend, not the 21-year average.
    Everything else: the mean m of the last 126 steps, shrunk toward 0 by its t-statistic t = m / (sd / sqrt(126))
    (James-Stein): drift = m x max(0, 1 - 4 / t^2). |t| <= 2 gives no drift at all; |t| = 4 keeps 75 %."""
    v = hist_steps(U, a, origin)
    if U["rules"][a]["drift"]:
        return float(v[-12:].mean()) if len(v) >= 12 else mu_window
    v = v[-126:]
    sd = v.std(ddof=1)
    t = v.mean() / (sd / np.sqrt(len(v))) if sd > 0 else 0.0
    return float(v.mean() * max(0.0, 1.0 - 4.0 / max(t * t, 1e-12)))

def m17_draws(U, origin, drift17=True, ewma_halflife=63):
    """M1.7 = M1.5 + the recent trend as drift (drift17). drift17=False is M1.5 (notebook 09)."""
    N = m0.N_DRAWS
    by_asset, x, log_mode, last = fit(U, origin, WINDOW)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
    sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])     # M1.5: EWMA sd, halflife 63
    sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    drift = {a: (mu[ai[a]] if U["rules"][a]["drift"] else 0.0) for a in by_asset}      # M1.5: drift by type
    if drift17:                                                                          # M1.7: the recent trend
        drift = {a: recent_trend(U, a, origin, mu[ai[a]]) for a in by_asset}
    def centre(a, h, d):
        return (0.0 if log_mode[a] else last[a]) + s[h] * d[a]
    mean = np.array([centre(a, h, drift) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((N, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"])
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(N, 1))), 1.0)
    sm = mean + dev
    out = np.empty((N, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(N, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

for u in ["t2-F3-divergence-2014", "t2-F4-cpi-vintage-2022", "t2-F1-ai-mom-2024", "t2-F2-cnh-tradewar-2018"]:
    U = UNITS_DATA[u]
    d = float(np.max(np.abs(m17_draws(U, U["asof"], drift17=False) - m13_draws(U, U["asof"], ewma_halflife=63))))
    print(f"check 1 — {u}: |m17_draws(drift17=False) - notebook 09's M1.5| = {d:.1e}")
    assert d == 0.0

def draw_variant(U, o, kw):
    return m17_draws(U, o, **kw)
VARIANTS = {"M1.5": dict(drift17=False), "M1.7": {}}
def score_unit(u):
    U = UNITS_DATA[u]
    out = []
    for o, y in BACKTEST[u]:
        for m, kw in VARIANTS.items():
            d = draw_variant(U, o, kw)
            out.append({"unit": u, "family": U["family"], "origin": o, "model": m, **parts(U["card"], d.reshape(d.shape[0], -1), y)})
    return out
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    G = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored {len(G) // len(VARIANTS):,} dates x {len(VARIANTS)} variants in {time.time() - t0:.0f}s")
SCV = {m: vs_m0(G[G.model == m]) for m in VARIANTS}
d = float((SCV["M1.5"] - vs_m0(pd.read_parquet(DATA / "m1_5_backtest_scores.parquet"))).abs().max())
print(f"check — the new pieces switched off vs notebook 09's saved M1.5: max |difference| per unit = {d:.1e}")
assert d < 1e-12
print(pd.Series({m: s.mean() for m, s in SCV.items()}, name="competition score vs M0 (all units)").round(4).to_string())
M17 = G[G.model == "M1.7"].drop(columns="model").reset_index(drop=True)

check 1 — t2-F3-divergence-2014: |m17_draws(drift17=False) - notebook 09's M1.5| = 0.0e+00
check 1 — t2-F4-cpi-vintage-2022: |m17_draws(drift17=False) - notebook 09's M1.5| = 0.0e+00
check 1 — t2-F1-ai-mom-2024: |m17_draws(drift17=False) - notebook 09's M1.5| = 0.0e+00
check 1 — t2-F2-cnh-tradewar-2018: |m17_draws(drift17=False) - notebook 09's M1.5| = 0.0e+00


scored 23,635 dates x 2 variants in 5s
check — the new pieces switched off vs notebook 09's saved M1.5: max |difference| per unit = 0.0e+00
M1.5   0.9444
M1.7   0.9312


## 2 — Worked example: the drift on four dates

For an asset without a drift rule: take its last 126 steps, their mean m and sd, t = m ÷ (sd ÷ √126), the weight
max(0, 1 − 4 ÷ t²), and the drift per step m × weight. The centre at h steps is `last + h × drift` (a log-stepped asset:
`last × exp(h × drift)`). CPI uses the mean of its last 12 monthly log steps instead.

In [3]:
def trend_parts(U, a, origin):
    v = hist_steps(U, a, origin)[-126:]
    m, sd = float(v.mean()), float(v.std(ddof=1))
    t = m / (sd / np.sqrt(len(v))) if sd > 0 else 0.0
    return m, sd, t, max(0.0, 1.0 - 4.0 / max(t * t, 1e-12))
def last_value(U, a, o):
    return U["rows"][a][bisect.bisect_right(U["dates"][a], o) - 1][1]

u = "t2-F2-cnh-tradewar-2018"; U = UNITS_DATA[u]; o, y = max(BACKTEST[u], key=lambda oy: abs(trend_parts(U, "CNY", oy[0])[2]))
m, sd, t, w = trend_parts(U, "CNY", o); h, last = U["steps"][64], last_value(U, "CNY", o)
display(pd.Series({"date": o, "last value": last, "m = mean of 126 log steps": m, "sd of the steps": sd, "t = m / (sd / √126)": t,
                   "weight = max(0, 1 - 4 / t²)": w, "drift per step = m × weight": m * w, "h (steps)": h,
                   "M1.5 centre = last": last, "M1.7 centre = last × exp(h × drift)": last * np.exp(h * m * w), "answer": y[0]},
                  name="(a) CNY, a crawling peg, on its strongest-trend date: t is large, so most of the trend is kept").to_frame())

u = "t2-F1-conflicting-texts-2024"; U = UNITS_DATA[u]
ts = sorted((abs(trend_parts(U, "UST_10Y", o)[2]), o, y[0]) for o, y in BACKTEST[u])
for label, (_, o, ans) in (("(b) UST_10Y on a typical date (the median |t|): noise, so no drift", ts[len(ts) // 2]),
                           ("(c) UST_10Y on its strongest-trend date", ts[-1])):
    m, sd, t, w = trend_parts(U, "UST_10Y", o); last = last_value(U, "UST_10Y", o)
    display(pd.Series({"date": o, "last value": last, "m = mean of 126 steps": m, "sd of the steps": sd, "t": t, "weight": w,
                       "drift per step": m * w, "h (steps)": 126, "M1.5 centre = last": last,
                       "M1.7 centre = last + 126 × drift": last + 126 * m * w, "answer at 126": ans}, name=label).to_frame())

u = "t2-F4-cpi-vintage-2022"; U = UNITS_DATA[u]; o, y = BACKTEST[u][-1]
_, x, _, _ = fit(U, o, WINDOW); v = hist_steps(U, "CPI_ALL", o); last = last_value(U, "CPI_ALL", o); s2 = U["steps"][21]
display(pd.Series({"date": o, "last value": last, "M1.5 drift: mean of the last 260 monthly log steps": x[0].mean(),
                   "  = % a year (x 12 x 100)": 1200 * x[0].mean(), "M1.7 drift: mean of the last 12": v[-12:].mean(),
                   "  = % a year": 1200 * v[-12:].mean(), "h (monthly steps)": s2,
                   "M1.5 centre = last × exp(h × 260-step drift)": last * np.exp(s2 * x[0].mean()),
                   "M1.7 centre = last × exp(h × 12-step drift)": last * np.exp(s2 * v[-12:].mean()), "answer": y[0]},
                  name="(d) CPI in late 2021: this year's inflation, not the 21-year average").to_frame())

rec = [{"asset type": asset_type(a), "|t|": abs(tp[2]), "weight": tp[3]}
       for u in UNIT_IDS for a in UNITS_DATA[u]["assets"] if not UNITS_DATA[u]["rules"][a]["drift"]
       for o, _ in BACKTEST[u] for tp in [trend_parts(UNITS_DATA[u], a, o)]]
print("how often M1.7 keeps a drift, over every backtest date:")
pd.DataFrame(rec).groupby("asset type").agg(dates=("weight", "size"), median_abs_t=("|t|", "median"),
    share_with_a_drift=("weight", lambda w: (w > 0).mean()), mean_weight=("weight", "mean"))

,"(a) CNY, a crawling peg, on its strongest-trend date: t is large, so most of the trend is kept"
date,2008-02-21
last value,7.139
m = mean of 126 log steps,-0.0004895
sd of the steps,0.001291
t = m / (sd / √126),-4.257
"weight = max(0, 1 - 4 / t²)",0.7793
drift per step = m × weight,-0.0003815
h (steps),64
M1.5 centre = last,7.139
M1.7 centre = last × exp(h × drift),6.967


,"(b) UST_10Y on a typical date (the median |t|): noise, so no drift"
date,2017-06-19
last value,2.19
m = mean of 126 steps,-0.003254
sd of the steps,0.03878
t,-0.9419
weight,0
drift per step,-0
h (steps),126
M1.5 centre = last,2.19
M1.7 centre = last + 126 × drift,2.19


,(c) UST_10Y on its strongest-trend date
date,2019-08-29
last value,1.5
m = mean of 126 steps,-0.01
sd of the steps,0.0399
t,-2.813
weight,0.4946
drift per step,-0.004946
h (steps),126
M1.5 centre = last,1.5
M1.7 centre = last + 126 × drift,0.8768


,"(d) CPI in late 2021: this year's inflation, not the 21-year average"
date,2021-10-01
last value,276.6
M1.5 drift: mean of the last 260 monthly log steps,0.001872
= % a year (x 12 x 100),2.246
M1.7 drift: mean of the last 12,0.005042
= % a year,6.05
h (monthly steps),2
M1.5 centre = last × exp(h × 260-step drift),277.6
M1.7 centre = last × exp(h × 12-step drift),279.4
answer,280.1


how often M1.7 keeps a drift, over every backtest date:


,dates,median_abs_t,share_with_a_drift,mean_weight
asset type,,,,
equity_factor,5633,0.8316,0.05308,0.01366
fx,17167,0.6703,0.05784,0.01736
rates,17354,0.7176,0.0835,0.02778
unemployment,12,0.4018,0.1667,0.0777


## 3 — Every model on the same backtest

In [4]:
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
models = {"M0": M0_SCORES, "M1.1 (per asset type)": M11_SCORES,
          "production walk (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          "M1.5 (EWMA sd 63)": pd.read_parquet(DATA / "m1_5_backtest_scores.parquet"),
          "M1.7 (recent trend)": M17}
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
production walk (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
M1.5 (EWMA sd 63),0.8932,0.99,0.9141,0.9642,0.9444,85/103
M1.7 (recent trend),0.8828,0.9656,0.9124,0.9505,0.9312,86/103


## 4 — M1.5 vs M1.7: per family, per asset type, and the units that moved most

In [5]:
PER_UNIT = pd.DataFrame({"family": fam, "asset type": pd.Series(TYPE), "M1.5 vs M0": SC["M1.5 (EWMA sd 63)"],
                         "M1.5 vs M0": SC["M1.5 (EWMA sd 63)"], "M1.7 vs M0": SC["M1.7 (recent trend)"]}).loc[:, lambda d: ~d.columns.duplicated()]
def summary(g):
    return {"units": len(g), "M1.5": g["M1.5 vs M0"].mean(), "M1.5": g["M1.5 vs M0"].mean(), "M1.7": g["M1.7 vs M0"].mean(),
            "M1.7 better than M1.5": f'{(g["M1.7 vs M0"] < g["M1.5 vs M0"]).sum()}/{len(g)}',
            "M1.7 better than M0": f'{(g["M1.7 vs M0"] < 1).sum()}/{len(g)}'}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)
moved = (PER_UNIT["M1.7 vs M0"] - PER_UNIT["M1.5 vs M0"]).sort_values()
print("the units that moved most (M1.7 - M1.5; below 0 = better):")
display(PER_UNIT.loc[list(moved.index[:8]) + list(moved.index[-5:])].assign(change=moved))

per family:


,units,M1.5,M1.7,M1.7 better than M1.5,M1.7 better than M0
family,,,,,
T2-F1,23,0.8932,0.8828,12/23,20/23
T2-F2,27,0.99,0.9656,11/27,19/27
T2-F3,22,0.9141,0.9124,9/22,22/22
T2-F4,31,0.9642,0.9505,7/31,25/31
all,103,0.9444,0.9312,39/103,86/103


per asset type:


,units,M1.5,M1.7,M1.7 better than M1.5,M1.7 better than M0
asset type,,,,,
cpi,2,0.899,0.7598,2/2,2/2
equity_factor,16,0.9665,0.9708,2/16,10/16
fx,35,0.9614,0.949,15/35,30/35
fx+rates,5,0.933,0.9298,3/5,5/5
payrolls,1,0.9682,0.6781,1/1,1/1
rates,43,0.9234,0.9135,16/43,38/43
unemployment,1,1.024,1.036,0/1,0/1


the units that moved most (M1.7 - M1.5; below 0 = better):


,family,asset type,M1.5 vs M0,M1.7 vs M0,change
t2-F2-cnh-tradewar-2018,T2-F2,fx,1.979,1.538,-0.4413
t2-F4-covid-nfp-2020,T2-F4,payrolls,0.9682,0.6781,-0.2902
t2-F4-cpi-vintage-2022,T2-F4,cpi,0.923,0.7487,-0.1743
t2-F2-powell-pivot-2019,T2-F2,rates,1.429,1.323,-0.1061
t2-F1-cpi-glidepath-2023,T2-F1,cpi,0.8749,0.7708,-0.1041
t2-F2-time-has-come-2024,T2-F2,rates,0.9863,0.919,-0.06737
t2-F1-pause-2006,T2-F1,rates,1.291,1.243,-0.04755
t2-F2-act-as-appropriate-2019,T2-F2,rates,0.9537,0.9063,-0.04733
t2-F2-whatever-it-takes-2012,T2-F2,fx,0.8385,0.853,0.01458
t2-F2-draghi-precommit-2014,T2-F2,fx,0.8298,0.8447,0.01494


## 5 — Save

In [6]:
M17.to_parquet(DATA / "m1_7_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m1_7_vs_m0_per_unit.csv"); FAMILY.to_csv(DATA / "m1_7_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m1_7_vs_m0_by_asset_type.csv"); LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for name in ("m1_7_backtest_scores.parquet", "m1_7_vs_m0_per_unit.csv", "m1_7_vs_m0_per_family.csv", "m1_7_vs_m0_by_asset_type.csv"):
    print(f"data/{name:<30} {(DATA / name).stat().st_size / 1e3:8.1f} kB")

data/m1_7_backtest_scores.parquet      867.1 kB
data/m1_7_vs_m0_per_unit.csv             7.8 kB
data/m1_7_vs_m0_per_family.csv           0.4 kB
data/m1_7_vs_m0_by_asset_type.csv        0.5 kB


## 6 — In short

In [7]:
for name in LADDER.index:
    print(f"  {name:<36} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
a = FAMILY.loc["all"]
print(f"M1.7 vs M1.5: better on {a['M1.7 better than M1.5']} units; per family M1.7 (M1.5): "
      + "; ".join(f"{f} {FAMILY.loc[f, 'M1.7']:.4f} ({FAMILY.loc[f, 'M1.5']:.4f})" for f in FAMILY.index if f != "all"))

  M0                                   1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)                0.9976   (better than M0 on 25/103)
  production walk (text off)           0.9726   (better than M0 on 74/103)
  M1.5 (EWMA sd 63)                    0.9444   (better than M0 on 85/103)
  M1.7 (recent trend)                  0.9312   (better than M0 on 86/103)
M1.7 vs M1.5: better on 39/103 units; per family M1.7 (M1.5): T2-F1 0.8828 (0.8932); T2-F2 0.9656 (0.9900); T2-F3 0.9124 (0.9141); T2-F4 0.9505 (0.9642)
